<a href="https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula12b_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup
!cat aclImdb/train/pos/4077_10.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  6458k      0  0:00:12  0:00:12 --:--:-- 13.9M
I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drin

In [2]:
import os, pathlib, shutil, random
base_dir = pathlib.Path("aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"
for category in ["pos", "neg"]:
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname, val_dir / category / fname)

In [3]:
from tensorflow import keras

batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [4]:
from tensorflow.keras.layers import TextVectorization
max_length = 600
max_tokens=20000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length)
text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

In [18]:
!pip install transformers

In [ ]:
from transformers import pipeline

pipe = pipeline("text-classification", model="lvwerra/distilbert-imdb")

In [22]:
results = pipe(["This is a great movie", "This is a bad movie"])
for result in results:
  print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

label: POSITIVE, with score: 0.9957
label: NEGATIVE, with score: 0.995


In [28]:
from sklearn.metrics import accuracy_score
import numpy as np
import time

y_pred = []
test_labels = []
start = time.time()
# Iterate over the original test_ds which yields (text_string_tensor, label_tensor)
for batch, (texts, targets) in enumerate(test_ds):
  # Process each individual text and its corresponding target label in the batch
  for i in range(len(texts)):
    # Extract the text string and decode it from bytes to utf-8 string
    current_text = texts[i].numpy().decode('utf-8')
    # Pass the string to the pipeline, truncating if necessary
    yp = pipe(current_text, truncation=True, max_length=512) # pipe expects a string or list of strings

    # Get the true label: targets[i] is a scalar tensor. Convert to int, 1 for positive, 0 for negative
    # Assuming `1` corresponds to positive class in `test_ds` (e.g., from text_dataset_from_directory)
    true_label = int(targets[i].numpy() == 1)
    test_labels.append(true_label)

    # Get the predicted label from the pipeline output
    # 'lvwerra/distilbert-imdb' outputs 'POSITIVE' or 'NEGATIVE'
    predicted_label_is_positive = (yp[0]['label'] == 'POSITIVE')
    y_pred.append(predicted_label_is_positive)

  print(f"{batch} {time.time()-start:.2f} s - Accuracy: {accuracy_score(test_labels, y_pred):.4f}")
print("Final Accuracy", accuracy_score(test_labels, y_pred))

0 0.43 s - Accuracy: 0.9688
1 0.76 s - Accuracy: 0.9531
2 1.12 s - Accuracy: 0.9583
3 1.52 s - Accuracy: 0.9297
4 1.87 s - Accuracy: 0.9313
5 2.24 s - Accuracy: 0.9271
6 2.60 s - Accuracy: 0.9196
7 2.96 s - Accuracy: 0.9102
8 3.31 s - Accuracy: 0.9097
9 3.70 s - Accuracy: 0.9062
10 4.08 s - Accuracy: 0.9062
11 4.45 s - Accuracy: 0.9089
12 4.80 s - Accuracy: 0.9135
13 5.19 s - Accuracy: 0.9152
14 5.51 s - Accuracy: 0.9167
15 5.86 s - Accuracy: 0.9160
16 6.15 s - Accuracy: 0.9191
17 6.54 s - Accuracy: 0.9149
18 6.86 s - Accuracy: 0.9178
19 7.20 s - Accuracy: 0.9141
20 7.56 s - Accuracy: 0.9152
21 7.89 s - Accuracy: 0.9162
22 8.24 s - Accuracy: 0.9144
23 8.58 s - Accuracy: 0.9167
24 8.96 s - Accuracy: 0.9163
25 9.33 s - Accuracy: 0.9147
26 9.72 s - Accuracy: 0.9097
27 10.04 s - Accuracy: 0.9129
28 10.42 s - Accuracy: 0.9159
29 10.76 s - Accuracy: 0.9177
30 11.09 s - Accuracy: 0.9163
31 11.46 s - Accuracy: 0.9160
32 11.77 s - Accuracy: 0.9167
33 12.16 s - Accuracy: 0.9182
34 12.50 s - Accu